In [ ]:
# P10: Credit Card Fraud Detection using Deep Learning
# Code by Parthiv Abhani

# ==============================
# 1. Import Libraries
# ==============================
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

print("TensorFlow Version:", tf.__version__)


# ==============================
# 2. Create Internal Dataset
# ==============================
np.random.seed(42)

n_samples = 10000
fraud_ratio = 0.02
n_fraud = int(n_samples * fraud_ratio)
n_normal = n_samples - n_fraud

# Normal transactions
normal_data = pd.DataFrame({
    "Amount": np.random.lognormal(mean=3.2, sigma=1.0, size=n_normal),
    "Time": np.random.uniform(0, 24, n_normal),
    "Location_Score": np.random.normal(0, 1, n_normal),
    "Transaction_Frequency": np.random.poisson(3, n_normal),
    "Device_Risk": np.random.uniform(0, 0.5, n_normal),
    "Merchant_Risk": np.random.uniform(0, 0.5, n_normal),
    "International": np.random.binomial(1, 0.05, n_normal)
})

# Fraudulent transactions
fraud_data = pd.DataFrame({
    "Amount": np.random.lognormal(mean=4.5, sigma=1.2, size=n_fraud),
    "Time": np.random.uniform(0, 24, n_fraud),
    "Location_Score": np.random.normal(2, 1.5, n_fraud),
    "Transaction_Frequency": np.random.poisson(8, n_fraud),
    "Device_Risk": np.random.uniform(0.5, 1.0, n_fraud),
    "Merchant_Risk": np.random.uniform(0.5, 1.0, n_fraud),
    "International": np.random.binomial(1, 0.5, n_fraud)
})

normal_data["Fraud"] = 0
fraud_data["Fraud"] = 1

data = pd.concat([normal_data, fraud_data], ignore_index=True)

# Shuffle dataset
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nDataset Shape:", data.shape)
print("\nClass Distribution:")
print(data["Fraud"].value_counts())


# ==============================
# 3. Separate Features and Target
# ==============================
X = data.drop("Fraud", axis=1)
y = data["Fraud"]

# Train-test split using stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Feature scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# ==============================
# 4. Handle Imbalanced Data
#    Using Class Weights
# ==============================
normal_count = np.sum(y_train == 0)
fraud_count = np.sum(y_train == 1)

total = normal_count + fraud_count

class_weight = {
    0: total / (2 * normal_count),
    1: total / (2 * fraud_count)
}

print("\nClass Weights:")
print(class_weight)


# ==============================
# 5. Build Deep Neural Network
# ==============================
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),

    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(16, activation="relu"),

    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


# ==============================
# 6. Train Model
# ==============================
history = model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.2,
    class_weight=class_weight,
    verbose=1
)


# ==============================
# 7. Evaluate Model
# ==============================
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

print("\nTest Accuracy:", round(accuracy * 100, 2), "%")


# ==============================
# 8. Generate Predictions
# ==============================
y_probability = model.predict(X_test)

# Convert probability into class
y_pred = (y_probability >= 0.5).astype(int).flatten()


# ==============================
# 9. Precision, Recall and F1
# ==============================
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\nEvaluation Metrics")
print("-------------------------")
print("Precision :", round(precision, 4))
print("Recall    :", round(recall, 4))
print("F1-Score  :", round(f1, 4))


# ==============================
# 10. Classification Report
# ==============================
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Normal", "Fraud"]
))


# ==============================
# 11. Confusion Matrix
# ==============================
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0, 1], ["Normal", "Fraud"])
plt.yticks([0, 1], ["Normal", "Fraud"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.show()


# ==============================
# 12. Training Graph
# ==============================
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()